In [ ]:
pip install qiskit_machine_learning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 54.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [qiskit_machine_learning]


In [ ]:
import pandas as pd

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import make_moons
# Generate the Make Moons dataset

X, y = make_moons(
    n_samples=200,
    noise=0.15,
    random_state=42
)

X = pd.DataFrame(X, columns=["Feature 1", "Feature 2"])
y = pd.Series(y, name="target")

print("Features:", list(X.columns))
print("Classes:", sorted(y.unique()))
print("Dataset shape:", X.shape)

# Split the data, preserving class proportions
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


Features: ['Feature 1', 'Feature 2']
Classes: [np.int64(0), np.int64(1)]
Dataset shape: (200, 2)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train_raw)
X_test= scaler.transform(X_test_raw)

In [ ]:
import numpy as np
angle_scaler=MinMaxScaler(feature_range=(0,np.pi))
X_train_q=angle_scaler.fit_transform(X_train_raw)
X_test_q=angle_scaler.transform(X_test_raw)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
classical_svm=SVC(kernel="rbf")
classical_svm.fit(X_train, y_train)

classical_pred=classical_svm.predict(X_test)
print(confusion_matrix(y_test, classical_pred))

[[24  1]
 [ 0 25]]


In [ ]:
from qiskit.circuit.library import ZFeatureMap, RealAmplitudes, ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

# Create a four-qubit quantum feature map
feature_map = ZFeatureMap(
    feature_dimension=2,
    reps=2,
)

/tmp/ipykernel_511/307960940.py:5: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._z_feature_map.ZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the z_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZFeatureMap(


In [ ]:
quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

# Compute the kernel matrices
K_train = quantum_kernel.evaluate(X_train_q)
K_test = quantum_kernel.evaluate(
    X_test_q,
    X_train_q
)


In [ ]:
# Train the SVM with the precomputed quantum kernel
quantum_svm = SVC(kernel="precomputed")
quantum_svm.fit(K_train, y_train)

quantum_pred = quantum_svm.predict(K_test)

print("\nQuantum SVM accuracy:",
      accuracy_score(y_test, quantum_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, quantum_pred))


Quantum SVM accuracy: 0.88

Confusion matrix:
[[25  0]
 [ 6 19]]
